In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/stanfordu/stanford-question-answering-dataset/dev-v1.1.json
/kaggle/input/datasets/stanfordu/stanford-question-answering-dataset/train-v1.1.json


In [8]:
import json
import re
import string
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from collections import defaultdict

from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
)

from IPython.display import display

In [9]:
answer_start = 24
answer_end = 30

In [10]:
MODEL_NAME = "distilbert-base-uncased"

In [11]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print(tokenizer)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [12]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

In [13]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [14]:
question = "What is the capital of France?"
context = "Paris is the capital of France."

answer = "France"
answer_start = context.find(answer)
answer_end = answer_start + len(answer)

encoding = tokenizer(
    question,
    context,
    truncation="only_second",
    max_length=128,
    return_offsets_mapping=True
)

input_ids = encoding["input_ids"]
offsets = encoding["offset_mapping"]
sequence_ids = encoding.sequence_ids()

print("Question:", question)
print("Context:", context)
print("Answer:", answer)

print("answer_start:", answer_start)
print("answer_end:", answer_end)

print("\nTokenization:")
for idx, (token_id, offset, seq_id) in enumerate(
    zip(input_ids, offsets, sequence_ids)
):
    token = tokenizer.convert_ids_to_tokens(token_id)

    print(
        f"{idx:2d} | {token:12s} | "
        f"offset={offset} | sequence_id={seq_id}"
    )

Question: What is the capital of France?
Context: Paris is the capital of France.
Answer: France
answer_start: 24
answer_end: 30

Tokenization:
 0 | [CLS]        | offset=(0, 0) | sequence_id=None
 1 | what         | offset=(0, 4) | sequence_id=0
 2 | is           | offset=(5, 7) | sequence_id=0
 3 | the          | offset=(8, 11) | sequence_id=0
 4 | capital      | offset=(12, 19) | sequence_id=0
 5 | of           | offset=(20, 22) | sequence_id=0
 6 | france       | offset=(23, 29) | sequence_id=0
 7 | ?            | offset=(29, 30) | sequence_id=0
 8 | [SEP]        | offset=(0, 0) | sequence_id=None
 9 | paris        | offset=(0, 5) | sequence_id=1
10 | is           | offset=(6, 8) | sequence_id=1
11 | the          | offset=(9, 12) | sequence_id=1
12 | capital      | offset=(13, 20) | sequence_id=1
13 | of           | offset=(21, 23) | sequence_id=1
14 | france       | offset=(24, 30) | sequence_id=1
15 | .            | offset=(30, 31) | sequence_id=1
16 | [SEP]        | offset=(0, 0

In [15]:
def align_answer_span(
    offsets,
    sequence_ids,
    answer_start,
    answer_end
):
    start_token = None
    end_token = None

    for token_idx, (start, end) in enumerate(offsets):

        # Chỉ xét token thuộc context.
        if sequence_ids[token_idx] != 1:
            continue

        # Bỏ qua special tokens và padding.
        if start == end:
            continue

        # Tìm token chứa ký tự bắt đầu đáp án.
        if start <= answer_start < end:
            start_token = token_idx

        # Tìm token chứa ký tự kết thúc đáp án.
        if start < answer_end <= end:
            end_token = token_idx

    # Kiểm tra đáp án có được ánh xạ đầy đủ không.
    if start_token is None or end_token is None:
        return None, None

    if start_token > end_token:
        return None, None

    return start_token, end_token

In [16]:
start_token, end_token = align_answer_span(
    offsets=offsets,
    sequence_ids=sequence_ids,
    answer_start=answer_start,
    answer_end=answer_end
)

print("Start token:", start_token)
print("End token:", end_token)

if start_token is not None:
    print(
        "Start token text:",
        tokenizer.convert_ids_to_tokens(
            input_ids[start_token]
        )
    )

    print(
        "End token text:",
        tokenizer.convert_ids_to_tokens(
            input_ids[end_token]
        )
    )

Start token: 14
End token: 14
Start token text: france
End token text: france


In [17]:
# Offset giả lập để kiểm tra logic.
test_offsets = [
    (0, 5),
    (5, 10),
    (10, 16)
]

# Tất cả đều là token thuộc context.
test_sequence_ids = [1, 1, 1]

test_start = 7
test_end = 13

start_idx, end_idx = align_answer_span(
    test_offsets,
    test_sequence_ids,
    test_start,
    test_end
)

print("Answer span:", (test_start, test_end))
print("Start token:", start_idx)
print("End token:", end_idx)

assert start_idx == 1
assert end_idx == 2

print("Boundary alignment: PASSED")

Answer span: (7, 13)
Start token: 1
End token: 2
Boundary alignment: PASSED


In [18]:
long_context = (
    "Paris is a beautiful city in France. "
    "It is known for its museums and architecture. "
    "The capital of France is Paris."
)

long_answer = "Paris"
long_answer_start = long_context.rfind(long_answer)
long_answer_end = (
    long_answer_start + len(long_answer)
)

truncated_encoding = tokenizer(
    "What is the capital of France?",
    long_context,
    truncation="only_second",
    max_length=16,
    return_offsets_mapping=True
)

truncated_offsets = truncated_encoding["offset_mapping"]
truncated_sequence_ids = truncated_encoding.sequence_ids()

print("Answer start:", long_answer_start)
print("Answer end:", long_answer_end)

start_idx, end_idx = align_answer_span(
    truncated_offsets,
    truncated_sequence_ids,
    long_answer_start,
    long_answer_end
)

print("Start token:", start_idx)
print("End token:", end_idx)

if start_idx is None:
    print("Answer is truncated or not present in this feature.")
else:
    print("Answer is available in this feature.")

Answer start: 108
Answer end: 113
Start token: None
End token: None
Answer is truncated or not present in this feature.


In [19]:
window_context = (
    "Paris is the capital of France. "
    "France is located in Western Europe. "
    "The country has many famous cities. "
    "The population of Paris is very large. "
    "Paris is also known for the Eiffel Tower."
)

window_answer = "Eiffel Tower"

window_answer_start = window_context.find(
    window_answer
)

window_answer_end = (
    window_answer_start + len(window_answer)
)

window_encoding = tokenizer(
    "What is Paris known for?",
    window_context,
    truncation="only_second",
    max_length=16,
    stride=4,
    return_overflowing_tokens=True,
    return_offsets_mapping=True
)

print("Number of windows:", len(window_encoding["input_ids"]))
print("Answer start:", window_answer_start)
print("Answer end:", window_answer_end)

Number of windows: 12
Answer start: 172
Answer end: 184


In [20]:
for window_idx in range(
    len(window_encoding["input_ids"])
):
    offsets = window_encoding["offset_mapping"][
        window_idx
    ]

    sequence_ids = window_encoding.sequence_ids(
        window_idx
    )

    start_idx, end_idx = align_answer_span(
        offsets,
        sequence_ids,
        window_answer_start,
        window_answer_end
    )

    print(f"\nWindow {window_idx}")
    print("Start token:", start_idx)
    print("End token:", end_idx)

    if start_idx is None:
        print("Answer is outside this window.")
    else:
        print("Answer is inside this window.")


Window 0
Start token: None
End token: None
Answer is outside this window.

Window 1
Start token: None
End token: None
Answer is outside this window.

Window 2
Start token: None
End token: None
Answer is outside this window.

Window 3
Start token: None
End token: None
Answer is outside this window.

Window 4
Start token: None
End token: None
Answer is outside this window.

Window 5
Start token: None
End token: None
Answer is outside this window.

Window 6
Start token: None
End token: None
Answer is outside this window.

Window 7
Start token: None
End token: None
Answer is outside this window.

Window 8
Start token: None
End token: None
Answer is outside this window.

Window 9
Start token: None
End token: None
Answer is outside this window.

Window 10
Start token: None
End token: None
Answer is outside this window.

Window 11
Start token: 10
End token: 13
Answer is inside this window.


In [23]:
def create_qa_labels(
    encoding,
    answer_start,
    answer_end,
    feature_idx=0
):
    offsets = encoding["offset_mapping"]

    # Lấy offset của feature tương ứng.
    if (
        len(offsets) > 0
        and isinstance(offsets[0][0], (list, tuple))
    ):
        offsets = offsets[feature_idx]

    sequence_ids = encoding.sequence_ids(feature_idx)

    start_token, end_token = align_answer_span(
        offsets,
        sequence_ids,
        answer_start,
        answer_end
    )

    if start_token is None or end_token is None:
        cls_index = encoding["input_ids"][
            feature_idx
        ].index(tokenizer.cls_token_id)

        return cls_index, cls_index

    return start_token, end_token

In [25]:
start_position, end_position = create_qa_labels(
    encoding,
    answer_start,
    answer_end
)

print("start_position:", start_position)
print("end_position:", end_position)

start_position: 14
end_position: 14
